# 🧠 Spatial-Symbolic Transformer for ARC

## Overview
This notebook implements a **position-aware transformer architecture** specifically designed for ARC matrix reasoning:
- **Input**: Matrix values with explicit (x, y) positions
- **Architecture**: Spatial-Symbolic Transformer with self-attention
- **Goal**: Native spatial reasoning for discrete symbolic matrices

## Key Innovation
Unlike CNN-based encoders that treat matrices as images, this architecture:
- Converts matrices to sequences of (value, x, y) tokens
- Uses self-attention to learn spatial relationships
- Handles variable matrix sizes naturally
- Provides explicit positional reasoning capabilities

In [ ]:
# Import required libraries
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
import math
import time
import warnings
from tqdm import tqdm
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"Available GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# Set matplotlib style
plt.style.use('default')
sns.set_palette("husl")

print("\n🚀 Spatial-Symbolic Transformer Environment Ready!")

In [ ]:
# Load ARC dataset (same as before)
def load_arc_data():
    """Load ARC training data from Kaggle input directory"""
    
    # Kaggle data paths
    train_challenges_path = '/kaggle/input/arc-prize-2025/arc-agi_training_challenges.json'
    train_solutions_path = '/kaggle/input/arc-prize-2025/arc-agi_training_solutions.json'
    
    try:
        # Load training challenges
        with open(train_challenges_path, 'r') as f:
            challenges = json.load(f)
        
        # Load training solutions
        with open(train_solutions_path, 'r') as f:
            solutions = json.load(f)
        
        print(f"✅ Loaded {len(challenges)} training tasks")
        print(f"✅ Loaded {len(solutions)} training solutions")
        
        return challenges, solutions
    
    except FileNotFoundError as e:
        print(f"❌ ARC data files not found: {e}")
        print("📁 Expected paths:")
        print(f"   • {train_challenges_path}")
        print(f"   • {train_solutions_path}")
        print("\n💡 Solutions:")
        print("   1. Upload this notebook to Kaggle with ARC Prize 2025 dataset")
        print("   2. Or modify the paths above to point to your local ARC data files")
        print("   3. Download ARC data from: https://www.kaggle.com/competitions/arc-prize-2025/data")
        
        raise FileNotFoundError("ARC dataset not found. Please check file paths or run on Kaggle.")

# Load the data
print("🔄 Loading ARC dataset...")
challenges, solutions = load_arc_data()

# Quick dataset verification
sample_task_id = list(challenges.keys())[0]
sample_task = challenges[sample_task_id]

print(f"\n📋 Sample Task ID: {sample_task_id}")
print(f"📊 Number of training examples: {len(sample_task['train'])}")
print(f"🧪 Number of test examples: {len(sample_task['test'])}")

# Show first training example structure
first_example = sample_task['train'][0]
input_grid = np.array(first_example['input'])
output_grid = np.array(first_example['output'])

print(f"\n🔍 First training example analysis:")
print(f"   Input shape: {input_grid.shape}")
print(f"   Output shape: {output_grid.shape}")
print(f"   Input unique values: {np.unique(input_grid)}")
print(f"   Output unique values: {np.unique(output_grid)}")

print(f"\n✅ Data loading successful! Ready for spatial-symbolic processing.")

In [ ]:
# Step 3: Matrix-to-Sequence Converter - Core Innovation
class MatrixToSequenceConverter:
    """
    Converts ARC matrices to sequences of (value, x, y) tokens for transformer processing.
    This is the key innovation that makes position explicit rather than implicit.
    """
    
    def __init__(self, normalize_positions=True):
        self.normalize_positions = normalize_positions
    
    def matrix_to_sequence(self, matrix):
        """
        Convert matrix to sequence of position-aware tokens.
        
        Args:
            matrix: numpy array of shape (height, width) with values 0-9
            
        Returns:
            tokens: list of (value, x, y) tuples
            metadata: dict with original dimensions and stats
        """
        height, width = matrix.shape
        tokens = []
        
        # Convert each cell to (value, x, y) token
        for y in range(height):
            for x in range(width):
                value = int(matrix[y, x])
                
                if self.normalize_positions:
                    # Normalize positions to [0, 1] range for better learning
                    norm_x = x / max(width - 1, 1)  # Avoid division by zero
                    norm_y = y / max(height - 1, 1)
                    tokens.append((value, norm_x, norm_y))
                else:
                    # Use absolute positions
                    tokens.append((value, x, y))
        
        # Metadata for reconstruction and analysis
        metadata = {
            'height': height,
            'width': width,
            'num_tokens': len(tokens),
            'unique_values': len(np.unique(matrix)),
            'total_cells': height * width
        }
        
        return tokens, metadata
    
    def sequence_to_matrix(self, tokens, target_height, target_width):
        """
        Convert sequence back to matrix (for reconstruction testing).
        
        Args:
            tokens: list of (value, x, y) tuples
            target_height, target_width: desired output dimensions
            
        Returns:
            matrix: reconstructed numpy array
        """
        matrix = np.zeros((target_height, target_width), dtype=int)
        
        for value, x, y in tokens:
            if self.normalize_positions:
                # Denormalize positions
                abs_x = int(round(x * max(target_width - 1, 1)))
                abs_y = int(round(y * max(target_height - 1, 1)))
            else:
                abs_x, abs_y = int(x), int(y)
            
            # Ensure coordinates are within bounds
            if 0 <= abs_y < target_height and 0 <= abs_x < target_width:
                matrix[abs_y, abs_x] = int(value)
        
        return matrix
    
    def visualize_conversion(self, matrix, tokens, metadata):
        """Visualize the matrix-to-sequence conversion process."""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        
        # Original matrix
        im1 = ax1.imshow(matrix, cmap='tab10', vmin=0, vmax=9)
        ax1.set_title(f'Original Matrix ({metadata["height"]}x{metadata["width"]})')
        ax1.set_xlabel('X coordinate')
        ax1.set_ylabel('Y coordinate')
        
        # Add grid and labels
        ax1.set_xticks(range(metadata['width']))
        ax1.set_yticks(range(metadata['height']))
        ax1.grid(True, alpha=0.3)
        
        # Token sequence visualization
        values = [token[0] for token in tokens]
        x_coords = [token[1] for token in tokens]
        y_coords = [token[2] for token in tokens]
        
        scatter = ax2.scatter(x_coords, y_coords, c=values, cmap='tab10', 
                            vmin=0, vmax=9, s=100, alpha=0.8)
        ax2.set_title(f'Token Sequence ({len(tokens)} tokens)')
        ax2.set_xlabel('Normalized X' if self.normalize_positions else 'X coordinate')
        ax2.set_ylabel('Normalized Y' if self.normalize_positions else 'Y coordinate')
        
        # Add colorbar
        plt.colorbar(scatter, ax=ax2, label='Value')
        
        plt.tight_layout()
        plt.show()
        
        return fig

# Test the Matrix-to-Sequence Converter
print("🔧 Testing Matrix-to-Sequence Conversion...")

# Initialize converter
converter = MatrixToSequenceConverter(normalize_positions=True)

# Test with the sample matrix we loaded
test_matrix = input_grid  # Use the first example from ARC data
print(f"\n📊 Testing with matrix shape: {test_matrix.shape}")
print(f"Matrix content:\n{test_matrix}")

# Convert to sequence
tokens, metadata = converter.matrix_to_sequence(test_matrix)

print(f"\n🎯 Conversion Results:")
print(f"   • Original matrix: {metadata['height']}x{metadata['width']} = {metadata['total_cells']} cells")
print(f"   • Token sequence: {metadata['num_tokens']} tokens")
print(f"   • Unique values: {metadata['unique_values']}")

# Show first few tokens
print(f"\n🔍 First 10 tokens (value, norm_x, norm_y):")
for i, token in enumerate(tokens[:10]):
    value, x, y = token
    print(f"   Token {i}: value={value}, x={x:.3f}, y={y:.3f}")

# Test reconstruction
reconstructed = converter.sequence_to_matrix(tokens, metadata['height'], metadata['width'])
reconstruction_accurate = np.array_equal(test_matrix, reconstructed)

print(f"\n✅ Reconstruction Test:")
print(f"   • Original == Reconstructed: {reconstruction_accurate}")
if reconstruction_accurate:
    print("   🎉 Perfect round-trip conversion!")
else:
    print("   ❌ Reconstruction error - need to debug")

# Visualize the conversion
print(f"\n📈 Visualizing conversion process...")
converter.visualize_conversion(test_matrix, tokens, metadata)

In [ ]:
# Step 4: Position-Value Joint Embedding
class PositionValueEmbedding(nn.Module):
    """
    Creates joint embeddings for (value, x, y) triplets.
    This combines discrete value embeddings with continuous position embeddings.
    """
    
    def __init__(self, embed_dim=128, max_seq_length=900):  # 30x30 = 900 max tokens
        super().__init__()
        self.embed_dim = embed_dim
        self.max_seq_length = max_seq_length
        
        # Value embedding: discrete values 0-9 → dense vectors
        self.value_embedding = nn.Embedding(10, embed_dim // 2)  # Half the dimensions
        
        # Position embeddings: continuous (x, y) → dense vectors  
        self.position_mlp = nn.Sequential(
            nn.Linear(2, embed_dim // 4),  # (x, y) → quarter dimensions
            nn.ReLU(),
            nn.Linear(embed_dim // 4, embed_dim // 2)  # → half dimensions
        )
        
        # Final projection to combine value + position
        self.final_projection = nn.Linear(embed_dim, embed_dim)
        
        # Layer normalization for stable training
        self.layer_norm = nn.LayerNorm(embed_dim)
        
        print(f"✅ PositionValueEmbedding initialized:")
        print(f"   • Embedding dimension: {embed_dim}")
        print(f"   • Value embedding: 10 → {embed_dim // 2}D")
        print(f"   • Position embedding: 2 → {embed_dim // 2}D") 
        print(f"   • Combined output: {embed_dim}D")
    
    def forward(self, token_sequence):
        """
        Embed sequence of (value, x, y) tokens.
        
        Args:
            token_sequence: tensor of shape (batch_size, seq_len, 3)
                          where each token is [value, x, y]
        
        Returns:
            embeddings: tensor of shape (batch_size, seq_len, embed_dim)
        """
        batch_size, seq_len, _ = token_sequence.shape
        
        # Extract components
        values = token_sequence[:, :, 0].long()  # (batch_size, seq_len)
        positions = token_sequence[:, :, 1:3]    # (batch_size, seq_len, 2)
        
        # Embed values (discrete)
        value_embeds = self.value_embedding(values)  # (batch_size, seq_len, embed_dim//2)
        
        # Embed positions (continuous)
        pos_embeds = self.position_mlp(positions)   # (batch_size, seq_len, embed_dim//2)
        
        # Concatenate value and position embeddings
        combined = torch.cat([value_embeds, pos_embeds], dim=-1)  # (batch_size, seq_len, embed_dim)
        
        # Final projection and normalization
        embeddings = self.final_projection(combined)
        embeddings = self.layer_norm(embeddings)
        
        return embeddings
    
    def get_embedding_info(self):
        """Return information about the embedding structure."""
        return {
            'total_params': sum(p.numel() for p in self.parameters()),
            'value_embedding_params': sum(p.numel() for p in self.value_embedding.parameters()),
            'position_mlp_params': sum(p.numel() for p in self.position_mlp.parameters()),
            'embed_dim': self.embed_dim
        }

# Test Position-Value Joint Embedding
print("🧠 Testing Position-Value Joint Embedding...")

# Initialize embedding layer
embed_dim = 128
embedding_layer = PositionValueEmbedding(embed_dim=embed_dim).to(device)

# Convert our test tokens to tensor format
def tokens_to_tensor(tokens, batch_size=1):
    """Convert list of (value, x, y) tuples to tensor format."""
    token_tensor = torch.tensor(tokens, dtype=torch.float32)
    
    # Add batch dimension if needed
    if batch_size > 1:
        # For batch processing, we'd need to pad sequences to same length
        # For now, just single batch
        token_tensor = token_tensor.unsqueeze(0)  # (1, seq_len, 3)
    else:
        token_tensor = token_tensor.unsqueeze(0)  # (1, seq_len, 3)
    
    return token_tensor.to(device)

# Test with our converted tokens
test_tokens_tensor = tokens_to_tensor(tokens)
print(f"\n📊 Input tensor shape: {test_tokens_tensor.shape}")
print(f"Input tensor content:\n{test_tokens_tensor}")

# Forward pass through embedding
with torch.no_grad():
    embeddings = embedding_layer(test_tokens_tensor)

print(f"\n🎯 Embedding Results:")
print(f"   • Input shape: {test_tokens_tensor.shape}")
print(f"   • Output shape: {embeddings.shape}")
print(f"   • Embedding dim: {embeddings.shape[-1]}")
print(f"   • Output range: [{embeddings.min():.3f}, {embeddings.max():.3f}]")

# Show embedding info
embed_info = embedding_layer.get_embedding_info()
print(f"\n📈 Embedding Layer Statistics:")
for key, value in embed_info.items():
    if 'params' in key:
        print(f"   • {key}: {value:,}")
    else:
        print(f"   • {key}: {value}")

print(f"\n✅ Position-Value Embedding test successful!")
print(f"🚀 Ready for Step 5: Spatial-Symbolic Transformer Blocks!")

In [ ]:
# Step 5: Spatial-Symbolic Transformer Block
class SpatialSymbolicTransformerBlock(nn.Module):
    """
    Transformer block specifically designed for spatial reasoning with discrete symbols.
    Uses multi-head self-attention to learn spatial relationships between positions.
    """
    
    def __init__(self, embed_dim=128, num_heads=8, ff_dim=512, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        
        # Multi-head self-attention for spatial relationships
        self.self_attention = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True  # Input format: (batch, seq, embed_dim)
        )
        
        # Feed-forward network for feature transformation
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, embed_dim),
            nn.Dropout(dropout)
        )
        
        # Layer normalization for residual connections
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        
        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)
        
        print(f"✅ SpatialSymbolicTransformerBlock initialized:")
        print(f"   • Embed dim: {embed_dim}")
        print(f"   • Num heads: {num_heads}")
        print(f"   • FF dim: {ff_dim}")
        print(f"   • Dropout: {dropout}")
    
    def forward(self, x, return_attention=False):
        """
        Forward pass through transformer block.
        
        Args:
            x: input embeddings (batch_size, seq_len, embed_dim)
            return_attention: whether to return attention weights
            
        Returns:
            output: transformed embeddings (batch_size, seq_len, embed_dim)
            attention_weights: if return_attention=True
        """
        # Self-attention with residual connection
        attn_output, attn_weights = self.self_attention(x, x, x)
        x = self.norm1(x + self.dropout(attn_output))
        
        # Feed-forward with residual connection
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        
        if return_attention:
            return x, attn_weights
        return x

class SpatialSymbolicTransformer(nn.Module):
    """
    Complete Spatial-Symbolic Transformer for ARC matrix reasoning.
    Stacks multiple transformer blocks for deep spatial understanding.
    """
    
    def __init__(self, embed_dim=128, num_heads=8, ff_dim=512, num_layers=6, 
                 dropout=0.1, output_dim=1024):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_layers = num_layers
        self.output_dim = output_dim
        
        # Position-Value embedding layer
        self.embedding = PositionValueEmbedding(embed_dim=embed_dim)
        
        # Stack of transformer blocks
        self.transformer_blocks = nn.ModuleList([
            SpatialSymbolicTransformerBlock(embed_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])
        
        # Global aggregation to fixed output size
        self.global_pool = nn.AdaptiveAvgPool1d(1)  # Pool across sequence dimension
        
        # Final projection to output dimension
        self.output_projection = nn.Sequential(
            nn.Linear(embed_dim, output_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(output_dim, output_dim)
        )
        
        # Layer normalization for final output
        self.output_norm = nn.LayerNorm(output_dim)
        
        print(f"✅ SpatialSymbolicTransformer initialized:")
        print(f"   • Input: Variable-size matrix → {output_dim}D features")
        print(f"   • Architecture: {num_layers} transformer blocks")
        print(f"   • Total parameters: {sum(p.numel() for p in self.parameters()):,}")
    
    def forward(self, token_sequence, return_attention=False):
        """
        Forward pass: matrix tokens → global features.
        
        Args:
            token_sequence: (batch_size, seq_len, 3) - (value, x, y) tokens
            return_attention: whether to return attention from last layer
            
        Returns:
            global_features: (batch_size, output_dim)
            attention_weights: if return_attention=True
        """
        # Embed tokens: (batch_size, seq_len, 3) → (batch_size, seq_len, embed_dim)
        x = self.embedding(token_sequence)
        
        # Pass through transformer blocks
        attention_weights = None
        for i, block in enumerate(self.transformer_blocks):
            if return_attention and i == len(self.transformer_blocks) - 1:
                x, attention_weights = block(x, return_attention=True)
            else:
                x = block(x)
        
        # Global aggregation: (batch_size, seq_len, embed_dim) → (batch_size, embed_dim)
        # Transpose for adaptive pooling: (batch_size, embed_dim, seq_len)
        x = x.transpose(1, 2)
        x = self.global_pool(x).squeeze(-1)  # (batch_size, embed_dim)
        
        # Final projection: (batch_size, embed_dim) → (batch_size, output_dim)
        global_features = self.output_projection(x)
        global_features = self.output_norm(global_features)
        
        if return_attention:
            return global_features, attention_weights
        return global_features
    
    def get_model_info(self):
        """Get detailed model information."""
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        
        return {
            'total_params': total_params,
            'trainable_params': trainable_params,
            'model_size_mb': total_params * 4 / (1024 * 1024),  # 4 bytes per float32
            'embed_dim': self.embed_dim,
            'num_layers': self.num_layers,
            'output_dim': self.output_dim
        }

# Test Spatial-Symbolic Transformer
print("🤖 Testing Complete Spatial-Symbolic Transformer...")

# Initialize the full transformer
transformer = SpatialSymbolicTransformer(
    embed_dim=128,
    num_heads=8,
    ff_dim=512,
    num_layers=6,
    output_dim=1024
).to(device)

# Test with our token sequence
print(f"\n📊 Testing with token sequence shape: {test_tokens_tensor.shape}")

# Forward pass
with torch.no_grad():
    global_features, attention_weights = transformer(test_tokens_tensor, return_attention=True)

print(f"\n🎯 Transformer Results:")
print(f"   • Input shape: {test_tokens_tensor.shape}")
print(f"   • Output shape: {global_features.shape}")
print(f"   • Feature range: [{global_features.min():.3f}, {global_features.max():.3f}]")
print(f"   • Attention shape: {attention_weights.shape if attention_weights is not None else 'None'}")

# Model information
model_info = transformer.get_model_info()
print(f"\n📈 Model Statistics:")
for key, value in model_info.items():
    if 'params' in key:
        print(f"   • {key}: {value:,}")
    elif 'size_mb' in key:
        print(f"   • {key}: {value:.1f} MB")
    else:
        print(f"   • {key}: {value}")

print(f"\n✅ Spatial-Symbolic Transformer test successful!")
print(f"🎉 We now have a complete position-aware encoder that converts matrices → 1024D features!")
print(f"🚀 Ready for Step 6: Dataset integration and reconstruction testing!")

In [ ]:
# Step 6: Reconstruction Decoder - Validate Feature Quality
class SpatialSymbolicDecoder(nn.Module):
    """
    Decoder that reconstructs matrices from 1024D features.
    This validates that our transformer features contain complete spatial information.
    """
    
    def __init__(self, feature_dim=1024, embed_dim=128, max_seq_length=900):
        super().__init__()
        self.feature_dim = feature_dim
        self.embed_dim = embed_dim
        self.max_seq_length = max_seq_length
        
        # Decoder network: features → sequence embeddings
        self.feature_to_embeddings = nn.Sequential(
            nn.Linear(feature_dim, embed_dim * 4),  # Expand features
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(embed_dim * 4, embed_dim * 2),
            nn.ReLU(),
            nn.Dropout(0.1)
        )
        
        # Learnable query tokens for different sequence lengths
        # We'll generate tokens for common matrix sizes
        self.position_queries = nn.Parameter(
            torch.randn(max_seq_length, 2) * 0.1  # (max_seq, 2) for (x,y) positions
        )
        
        # Cross-attention: features attend to position queries
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=8,
            dropout=0.1,
            batch_first=True
        )
        
        # Value prediction head: embeddings → color values (0-9)
        self.value_predictor = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(embed_dim, 10)  # 10 classes for colors 0-9
        )
        
        # Layer normalization
        self.norm = nn.LayerNorm(embed_dim)
        
        print(f"✅ SpatialSymbolicDecoder initialized:")
        print(f"   • Input: {feature_dim}D features → Variable-size matrix")
        print(f"   • Max sequence length: {max_seq_length}")
        print(f"   • Cross-attention heads: 8")
    
    def forward(self, global_features, target_shape):
        """
        Decode global features back to matrix.
        
        Args:
            global_features: (batch_size, feature_dim) - encoded features
            target_shape: (height, width) - desired output matrix size
            
        Returns:
            predicted_tokens: (batch_size, seq_len, 10) - logits for each position
            positions: (seq_len, 2) - (x, y) positions for each token
        """
        batch_size = global_features.size(0)
        height, width = target_shape
        seq_len = height * width
        
        # Generate position queries for target shape
        positions = []
        for y in range(height):
            for x in range(width):
                norm_x = x / max(width - 1, 1)
                norm_y = y / max(height - 1, 1)
                positions.append([norm_x, norm_y])
        
        positions = torch.tensor(positions, dtype=torch.float32, device=global_features.device)
        
        # Expand features to embeddings: (batch_size, feature_dim) → (batch_size, embed_dim * 2)
        expanded_features = self.feature_to_embeddings(global_features)
        
        # Reshape to sequence format: (batch_size, 2, embed_dim)
        key_value_features = expanded_features.view(batch_size, 2, self.embed_dim)
        
        # Position queries as queries: (seq_len, 2) → (seq_len, embed_dim)
        # Create a simple linear projection of positions to full embedding dimension
        pos_expanded = positions.unsqueeze(-1).expand(-1, -1, self.embed_dim // 4)  # (seq_len, 2, embed_dim//4)
        pos_flattened = pos_expanded.reshape(seq_len, -1)  # (seq_len, embed_dim//2)
        
        # Pad to full embedding dimension
        pos_embeddings = torch.cat([
            pos_flattened,
            torch.zeros(seq_len, self.embed_dim // 2, device=positions.device)
        ], dim=-1)
        
        # Add batch dimension: (1, seq_len, embed_dim) → (batch_size, seq_len, embed_dim)
        pos_embeddings = pos_embeddings.unsqueeze(0).expand(batch_size, -1, -1)
        
        # Cross-attention: position queries attend to global features
        attended_embeddings, attention_weights = self.cross_attention(
            pos_embeddings,  # queries: where we want to decode
            key_value_features,  # keys: what information we have
            key_value_features   # values: what information we have
        )
        
        # Normalize
        attended_embeddings = self.norm(attended_embeddings)
        
        # Predict values: (batch_size, seq_len, embed_dim) → (batch_size, seq_len, 10)
        predicted_logits = self.value_predictor(attended_embeddings)
        
        return predicted_logits, positions
    
    def predict_matrix(self, global_features, target_shape):
        """
        Predict complete matrix from global features.
        
        Args:
            global_features: (batch_size, feature_dim)
            target_shape: (height, width)
            
        Returns:
            predicted_matrix: (batch_size, height, width) - predicted values
            confidence: (batch_size, height, width) - prediction confidence
        """
        predicted_logits, positions = self.forward(global_features, target_shape)
        
        # Get predicted classes: (batch_size, seq_len)
        predicted_classes = predicted_logits.argmax(dim=-1)
        
        # Get confidence scores: (batch_size, seq_len)
        predicted_probs = F.softmax(predicted_logits, dim=-1)
        confidence_scores = predicted_probs.max(dim=-1)[0]
        
        # Reshape to matrix format
        height, width = target_shape
        predicted_matrix = predicted_classes.view(-1, height, width)
        confidence = confidence_scores.view(-1, height, width)
        
        return predicted_matrix, confidence

class SpatialSymbolicAutoEncoder(nn.Module):
    """
    Complete autoencoder: Matrix → Features → Matrix
    Combines the transformer encoder with reconstruction decoder.
    """
    
    def __init__(self, embed_dim=128, num_heads=8, ff_dim=512, num_layers=6, 
                 dropout=0.1, feature_dim=1024):
        super().__init__()
        
        # Encoder: Matrix → Features
        self.encoder = SpatialSymbolicTransformer(
            embed_dim=embed_dim,
            num_heads=num_heads, 
            ff_dim=ff_dim,
            num_layers=num_layers,
            dropout=dropout,
            output_dim=feature_dim
        )
        
        # Decoder: Features → Matrix
        self.decoder = SpatialSymbolicDecoder(
            feature_dim=feature_dim,
            embed_dim=embed_dim
        )
        
        self.converter = MatrixToSequenceConverter(normalize_positions=True)
        
        print(f"✅ SpatialSymbolicAutoEncoder initialized:")
        print(f"   • Total parameters: {sum(p.numel() for p in self.parameters()):,}")
    
    def forward(self, matrices, return_features=False):
        """
        Full autoencoder forward pass.
        
        Args:
            matrices: list of numpy arrays or tensor (batch_size, max_h, max_w)
            return_features: whether to return intermediate features
            
        Returns:
            reconstructed: (batch_size, height, width) - reconstructed matrices
            features: (batch_size, feature_dim) - if return_features=True
        """
        # Handle different input formats
        if isinstance(matrices, list):
            # Convert list of matrices to batch
            batch_tokens = []
            shapes = []
            
            for matrix in matrices:
                if isinstance(matrix, torch.Tensor):
                    matrix = matrix.cpu().numpy()
                
                tokens, metadata = self.converter.matrix_to_sequence(matrix)
                batch_tokens.append(torch.tensor(tokens, dtype=torch.float32))
                shapes.append((metadata['height'], metadata['width']))
            
            # For simplicity, use first matrix shape for all
            target_shape = shapes[0]
            
            # Pad sequences to same length
            max_len = max(len(tokens) for tokens in batch_tokens)
            padded_batch = []
            
            for tokens in batch_tokens:
                if len(tokens) < max_len:
                    # Pad with zeros
                    padding = torch.zeros(max_len - len(tokens), 3)
                    tokens = torch.cat([tokens, padding], dim=0)
                padded_batch.append(tokens)
            
            token_batch = torch.stack(padded_batch).to(next(self.parameters()).device)
        
        else:
            # Assume single matrix for now
            raise NotImplementedError("Tensor batch input not implemented yet")
        
        # Encode: matrices → features
        features = self.encoder(token_batch)
        
        # Decode: features → matrices
        reconstructed, confidence = self.decoder.predict_matrix(features, target_shape)
        
        if return_features:
            return reconstructed, features, confidence
        return reconstructed, confidence

# Test the Reconstruction Decoder
print("🔄 Testing Spatial-Symbolic Reconstruction Decoder...")

# Initialize the complete autoencoder
autoencoder = SpatialSymbolicAutoEncoder(
    embed_dim=128,
    num_heads=8,
    ff_dim=512,
    num_layers=6,
    feature_dim=1024
).to(device)

# Test with our sample matrix
test_matrices = [test_matrix]  # Use the 2x2 matrix from earlier

print(f"\n📊 Testing reconstruction on matrix shape: {test_matrix.shape}")
print(f"Original matrix:\n{test_matrix}")

# Forward pass through complete autoencoder
with torch.no_grad():
    reconstructed, features, confidence = autoencoder(test_matrices, return_features=True)

# Convert to numpy for analysis
reconstructed_np = reconstructed[0].cpu().numpy()
confidence_np = confidence[0].cpu().numpy()
features_np = features[0].cpu().numpy()

print(f"\n🎯 Reconstruction Results:")
print(f"   • Original shape: {test_matrix.shape}")
print(f"   • Reconstructed shape: {reconstructed_np.shape}")
print(f"   • Features shape: {features_np.shape}")
print(f"   • Feature range: [{features_np.min():.3f}, {features_np.max():.3f}]")

print(f"\n📋 Matrix Comparison:")
print(f"Original:\n{test_matrix}")
print(f"Reconstructed:\n{reconstructed_np}")
print(f"Confidence:\n{confidence_np}")

# Calculate reconstruction accuracy
reconstruction_accuracy = (test_matrix == reconstructed_np).mean()
print(f"\n✅ Reconstruction Accuracy: {reconstruction_accuracy:.3f} ({reconstruction_accuracy*100:.1f}%)")

# Calculate confidence statistics
avg_confidence = confidence_np.mean()
min_confidence = confidence_np.min()
print(f"📊 Confidence Statistics:")
print(f"   • Average confidence: {avg_confidence:.3f}")
print(f"   • Minimum confidence: {min_confidence:.3f}")

if reconstruction_accuracy == 1.0:
    print(f"🎉 Perfect reconstruction! The autoencoder works correctly!")
else:
    print(f"⚠️  Imperfect reconstruction - may need training or architecture adjustment")

print(f"\n✅ Reconstruction decoder test complete!")
print(f"🚀 Ready to train the autoencoder and compare with Stage 1 baselines!")

In [ ]:
# Step 7: Autoencoder Training - Validate Feature Learning
class ARCMatrixDataset(Dataset):
    """
    Dataset class for training the autoencoder on ARC matrices.
    Prepares matrices for reconstruction training.
    """
    
    def __init__(self, challenges, max_samples=1000, min_size=2, max_size=10):
        self.matrices = []
        self.converter = MatrixToSequenceConverter(normalize_positions=True)
        self.max_samples = max_samples
        
        print(f"🔄 Building ARC Matrix Dataset...")
        
        # Collect matrices from ARC training data
        sample_count = 0
        for task_id, task in challenges.items():
            if sample_count >= max_samples:
                break
                
            # Get matrices from training examples
            for example in task['train']:
                # Input matrix
                input_matrix = np.array(example['input'])
                if min_size <= input_matrix.shape[0] <= max_size and min_size <= input_matrix.shape[1] <= max_size:
                    self.matrices.append(input_matrix)
                    sample_count += 1
                    
                # Output matrix
                output_matrix = np.array(example['output'])
                if min_size <= output_matrix.shape[0] <= max_size and min_size <= output_matrix.shape[1] <= max_size:
                    self.matrices.append(output_matrix)
                    sample_count += 1
                
                if sample_count >= max_samples:
                    break
        
        print(f"✅ Collected {len(self.matrices)} matrices for training")
        
        # Analyze dataset statistics
        shapes = [matrix.shape for matrix in self.matrices]
        sizes = [shape[0] * shape[1] for shape in shapes]
        unique_values = [len(np.unique(matrix)) for matrix in self.matrices]
        
        print(f"📊 Dataset Statistics:")
        print(f"   • Matrix count: {len(self.matrices)}")
        print(f"   • Size range: {min(sizes)}-{max(sizes)} cells")
        print(f"   • Avg unique values: {np.mean(unique_values):.1f}")
        print(f"   • Most common shapes: {Counter(shapes).most_common(3)}")
    
    def __len__(self):
        return len(self.matrices)
    
    def __getitem__(self, idx):
        """
        Get a single matrix and convert to token format.
        Returns both the matrix and its tokenized version.
        """
        matrix = self.matrices[idx]
        
        # Convert to tokens
        tokens, metadata = self.converter.matrix_to_sequence(matrix)
        tokens_tensor = torch.tensor(tokens, dtype=torch.float32)
        
        return {
            'matrix': torch.tensor(matrix, dtype=torch.long),
            'tokens': tokens_tensor,
            'shape': matrix.shape,
            'metadata': metadata
        }

def collate_matrices(batch):
    """
    Custom collate function to handle variable-size matrices.
    Pads token sequences to the same length within each batch.
    """
    # Find max sequence length in this batch
    max_seq_len = max(item['tokens'].shape[0] for item in batch)
    
    # Prepare batch tensors
    batch_tokens = []
    batch_matrices = []
    batch_shapes = []
    batch_metadata = []
    
    for item in batch:
        tokens = item['tokens']
        
        # Pad tokens to max length
        if tokens.shape[0] < max_seq_len:
            padding = torch.zeros(max_seq_len - tokens.shape[0], 3)
            tokens = torch.cat([tokens, padding], dim=0)
        
        batch_tokens.append(tokens)
        batch_matrices.append(item['matrix'])
        batch_shapes.append(item['shape'])
        batch_metadata.append(item['metadata'])
    
    return {
        'tokens': torch.stack(batch_tokens),
        'matrices': batch_matrices,  # Keep as list due to variable shapes
        'shapes': batch_shapes,
        'metadata': batch_metadata
    }

def train_autoencoder(autoencoder, dataloader, num_epochs=5, learning_rate=1e-4):
    """
    Train the spatial-symbolic autoencoder on ARC matrices.
    """
    print(f"🏋️ Training Spatial-Symbolic Autoencoder...")
    print(f"   • Epochs: {num_epochs}")
    print(f"   • Learning rate: {learning_rate}")
    print(f"   • Dataset size: {len(dataloader.dataset)}")
    print(f"   • Batch size: {dataloader.batch_size}")
    
    # Setup optimizer and loss
    optimizer = torch.optim.Adam(autoencoder.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss(ignore_index=-1)  # Ignore padding tokens
    
    # Training loop
    autoencoder.train()
    epoch_losses = []
    epoch_accuracies = []
    
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        epoch_correct = 0
        epoch_total = 0
        
        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
        
        for batch_idx, batch in enumerate(progress_bar):
            # Get batch data
            tokens = batch['tokens'].to(device)  # (batch_size, seq_len, 3)
            matrices = batch['matrices']
            shapes = batch['shapes']
            
            # Forward pass through encoder
            features = autoencoder.encoder(tokens)
            
            # For each item in batch, decode and compute loss
            batch_loss = 0.0
            batch_correct = 0
            batch_total = 0
            
            for i in range(len(matrices)):
                matrix = matrices[i].to(device)
                shape = shapes[i]
                
                # Decode features for this specific shape
                predicted_logits, _ = autoencoder.decoder(features[i:i+1], shape)
                
                # Prepare targets: flatten matrix to sequence
                target_sequence = matrix.flatten()
                
                # Compute loss for this item
                item_loss = criterion(predicted_logits.squeeze(0), target_sequence)
                batch_loss += item_loss
                
                # Compute accuracy for this item
                predicted_classes = predicted_logits.argmax(dim=-1).squeeze(0)
                item_correct = (predicted_classes == target_sequence).sum().item()
                batch_correct += item_correct
                batch_total += target_sequence.numel()
            
            # Average loss over batch
            batch_loss = batch_loss / len(matrices)
            
            # Backward pass
            optimizer.zero_grad()
            batch_loss.backward()
            optimizer.step()
            
            # Update statistics
            epoch_loss += batch_loss.item()
            epoch_correct += batch_correct
            epoch_total += batch_total
            
            # Update progress bar
            current_acc = batch_correct / batch_total if batch_total > 0 else 0
            progress_bar.set_postfix({
                'loss': f'{batch_loss.item():.4f}',
                'acc': f'{current_acc:.3f}'
            })
        
        # Epoch statistics
        avg_loss = epoch_loss / len(dataloader)
        avg_accuracy = epoch_correct / epoch_total if epoch_total > 0 else 0
        
        epoch_losses.append(avg_loss)
        epoch_accuracies.append(avg_accuracy)
        
        print(f"📊 Epoch {epoch+1} Results:")
        print(f"   • Average Loss: {avg_loss:.4f}")
        print(f"   • Reconstruction Accuracy: {avg_accuracy:.3f} ({avg_accuracy*100:.1f}%)")
    
    return epoch_losses, epoch_accuracies

def evaluate_reconstruction_quality(autoencoder, dataset, num_samples=10):
    """
    Evaluate reconstruction quality on sample matrices.
    """
    print(f"\n🔍 Evaluating Reconstruction Quality on {num_samples} samples...")
    
    autoencoder.eval()
    total_accuracy = 0.0
    total_confidence = 0.0
    
    # Test on random samples
    sample_indices = np.random.choice(len(dataset), min(num_samples, len(dataset)), replace=False)
    
    with torch.no_grad():
        for i, idx in enumerate(sample_indices):
            sample = dataset[idx]
            matrix = sample['matrix'].numpy()
            
            # Reconstruct through autoencoder
            test_matrices = [matrix]
            reconstructed, features, confidence = autoencoder(test_matrices, return_features=True)
            
            # Calculate metrics
            reconstructed_np = reconstructed[0].cpu().numpy()
            confidence_np = confidence[0].cpu().numpy()
            
            accuracy = (matrix == reconstructed_np).mean()
            avg_confidence = confidence_np.mean()
            
            total_accuracy += accuracy
            total_confidence += avg_confidence
            
            # Show detailed results for first few samples
            if i < 3:
                print(f"\n📋 Sample {i+1} (Shape: {matrix.shape}):")
                print(f"   Original:\n{matrix}")
                print(f"   Reconstructed:\n{reconstructed_np}")
                print(f"   Accuracy: {accuracy:.3f} ({accuracy*100:.1f}%)")
                print(f"   Avg Confidence: {avg_confidence:.3f}")
    
    # Overall statistics
    avg_accuracy = total_accuracy / len(sample_indices)
    avg_confidence = total_confidence / len(sample_indices)
    
    print(f"\n🎯 Overall Reconstruction Quality:")
    print(f"   • Average Accuracy: {avg_accuracy:.3f} ({avg_accuracy*100:.1f}%)")
    print(f"   • Average Confidence: {avg_confidence:.3f}")
    
    return avg_accuracy, avg_confidence

# Build the training dataset
from collections import Counter
import random

print("🔄 Setting up training pipeline...")

# Create dataset
train_dataset = ARCMatrixDataset(
    challenges=challenges,
    max_samples=500,  # Start with smaller dataset for testing
    min_size=2,
    max_size=8
)

# Create dataloader
train_dataloader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_matrices
)

print(f"\n🏋️ Ready to train autoencoder!")
print(f"   • Dataset: {len(train_dataset)} matrices")
print(f"   • Batches: {len(train_dataloader)}")
print(f"   • Model parameters: {sum(p.numel() for p in autoencoder.parameters()):,}")

In [ ]:
# Execute Training and Evaluation
print("🚀 Starting Autoencoder Training...")

# Train the autoencoder
losses, accuracies = train_autoencoder(
    autoencoder=autoencoder,
    dataloader=train_dataloader,
    num_epochs=3,  # Start with 3 epochs for testing
    learning_rate=1e-4
)

print(f"\n📈 Training Complete!")

# Evaluate reconstruction quality
final_accuracy, final_confidence = evaluate_reconstruction_quality(
    autoencoder=autoencoder,
    dataset=train_dataset,
    num_samples=10
)

# Test on our original sample matrix again
print(f"\n🔄 Re-testing original sample after training...")
test_matrices = [test_matrix]

with torch.no_grad():
    reconstructed_trained, features_trained, confidence_trained = autoencoder(test_matrices, return_features=True)

reconstructed_trained_np = reconstructed_trained[0].cpu().numpy()
confidence_trained_np = confidence_trained[0].cpu().numpy()
trained_accuracy = (test_matrix == reconstructed_trained_np).mean()

print(f"📊 Original Sample Results After Training:")
print(f"   Original:\n{test_matrix}")
print(f"   Reconstructed:\n{reconstructed_trained_np}")
print(f"   Accuracy: {trained_accuracy:.3f} ({trained_accuracy*100:.1f}%)")
print(f"   Avg Confidence: {confidence_trained_np.mean():.3f}")

# Compare with pre-training results
print(f"\n🔄 Training Impact Analysis:")
print(f"   • Pre-training accuracy: 0.000 (0.0%)")
print(f"   • Post-training accuracy: {trained_accuracy:.3f} ({trained_accuracy*100:.1f}%)")
print(f"   • Improvement: {trained_accuracy*100:.1f} percentage points")
print(f"   • Average dataset accuracy: {final_accuracy*100:.1f}%")

if final_accuracy > 0.5:
    print(f"🎉 Excellent! The autoencoder learned meaningful spatial features!")
    print(f"✅ Ready to proceed with dual encoder architecture!")
elif final_accuracy > 0.2:
    print(f"✅ Good progress! Features are learning spatial patterns.")
    print(f"🔧 Consider more training epochs or architecture refinements.")
else:
    print(f"⚠️ Low accuracy suggests need for more training or architecture adjustments.")
    print(f"🔧 Consider: longer training, different learning rates, or data augmentation.")

print(f"\n🏁 Step 7 Complete: Autoencoder training and validation finished!")
print(f"🚀 Next: Dual encoder setup for input→output transformation learning!")

In [ ]:
# Step 8: Dual Encoder Architecture - Learn Input→Output Transformations
class ARCTransformationDataset(Dataset):
    """
    Dataset for learning input→output transformations in ARC tasks.
    Each sample contains an input matrix and its corresponding output matrix.
    """
    
    def __init__(self, challenges, solutions, max_samples=1000, min_size=2, max_size=10):
        self.transformation_pairs = []
        self.converter = MatrixToSequenceConverter(normalize_positions=True)
        
        print(f"🔄 Building ARC Transformation Dataset...")
        
        # Collect input→output pairs from ARC training data
        sample_count = 0
        for task_id, task in challenges.items():
            if sample_count >= max_samples:
                break
                
            # Get transformation examples from each task
            for example in task['train']:
                input_matrix = np.array(example['input'])
                output_matrix = np.array(example['output'])
                
                # Filter by size constraints
                if (min_size <= input_matrix.shape[0] <= max_size and 
                    min_size <= input_matrix.shape[1] <= max_size and
                    min_size <= output_matrix.shape[0] <= max_size and 
                    min_size <= output_matrix.shape[1] <= max_size):
                    
                    self.transformation_pairs.append({
                        'input': input_matrix,
                        'output': output_matrix,
                        'task_id': task_id
                    })
                    sample_count += 1
                    
                if sample_count >= max_samples:
                    break
        
        print(f"✅ Collected {len(self.transformation_pairs)} transformation pairs")
        
        # Analyze transformation dataset
        input_shapes = [pair['input'].shape for pair in self.transformation_pairs]
        output_shapes = [pair['output'].shape for pair in self.transformation_pairs]
        same_shape = sum(1 for i, o in zip(input_shapes, output_shapes) if i == o)
        
        print(f"📊 Transformation Dataset Statistics:")
        print(f"   • Total pairs: {len(self.transformation_pairs)}")
        print(f"   • Same shape transformations: {same_shape} ({same_shape/len(self.transformation_pairs)*100:.1f}%)")
        print(f"   • Different shape transformations: {len(self.transformation_pairs)-same_shape}")
        print(f"   • Most common input shapes: {Counter(input_shapes).most_common(3)}")
    
    def __len__(self):
        return len(self.transformation_pairs)
    
    def __getitem__(self, idx):
        pair = self.transformation_pairs[idx]
        
        # Convert both matrices to token format
        input_tokens, input_metadata = self.converter.matrix_to_sequence(pair['input'])
        output_tokens, output_metadata = self.converter.matrix_to_sequence(pair['output'])
        
        return {
            'input_matrix': torch.tensor(pair['input'], dtype=torch.long),
            'output_matrix': torch.tensor(pair['output'], dtype=torch.long),
            'input_tokens': torch.tensor(input_tokens, dtype=torch.float32),
            'output_tokens': torch.tensor(output_tokens, dtype=torch.float32),
            'input_shape': pair['input'].shape,
            'output_shape': pair['output'].shape,
            'task_id': pair['task_id']
        }

def collate_transformations(batch):
    """
    Custom collate function for transformation pairs.
    Handles variable-size input and output matrices.
    """
    # Find max sequence lengths
    max_input_len = max(item['input_tokens'].shape[0] for item in batch)
    max_output_len = max(item['output_tokens'].shape[0] for item in batch)
    
    # Prepare batch tensors
    batch_input_tokens = []
    batch_output_tokens = []
    batch_input_matrices = []
    batch_output_matrices = []
    batch_input_shapes = []
    batch_output_shapes = []
    batch_task_ids = []
    
    for item in batch:
        # Pad input tokens
        input_tokens = item['input_tokens']
        if input_tokens.shape[0] < max_input_len:
            padding = torch.zeros(max_input_len - input_tokens.shape[0], 3)
            input_tokens = torch.cat([input_tokens, padding], dim=0)
        
        # Pad output tokens
        output_tokens = item['output_tokens']
        if output_tokens.shape[0] < max_output_len:
            padding = torch.zeros(max_output_len - output_tokens.shape[0], 3)
            output_tokens = torch.cat([output_tokens, padding], dim=0)
        
        batch_input_tokens.append(input_tokens)
        batch_output_tokens.append(output_tokens)
        batch_input_matrices.append(item['input_matrix'])
        batch_output_matrices.append(item['output_matrix'])
        batch_input_shapes.append(item['input_shape'])
        batch_output_shapes.append(item['output_shape'])
        batch_task_ids.append(item['task_id'])
    
    return {
        'input_tokens': torch.stack(batch_input_tokens),
        'output_tokens': torch.stack(batch_output_tokens),
        'input_matrices': batch_input_matrices,
        'output_matrices': batch_output_matrices,
        'input_shapes': batch_input_shapes,
        'output_shapes': batch_output_shapes,
        'task_ids': batch_task_ids
    }

class DualSpatialSymbolicTransformer(nn.Module):
    """
    Dual encoder architecture for learning input→output transformations.
    Uses separate encoders for input and output, then learns transformation mapping.
    """
    
    def __init__(self, embed_dim=128, num_heads=8, ff_dim=512, num_layers=6, 
                 dropout=0.1, feature_dim=1024):
        super().__init__()
        
        # Input encoder: learns to encode input matrices
        self.input_encoder = SpatialSymbolicTransformer(
            embed_dim=embed_dim,
            num_heads=num_heads,
            ff_dim=ff_dim,
            num_layers=num_layers,
            dropout=dropout,
            output_dim=feature_dim
        )
        
        # Output encoder: learns to encode output matrices
        self.output_encoder = SpatialSymbolicTransformer(
            embed_dim=embed_dim,
            num_heads=num_heads,
            ff_dim=ff_dim,
            num_layers=num_layers,
            dropout=dropout,
            output_dim=feature_dim
        )
        
        # Transformation predictor: input features → output features
        self.transformation_predictor = nn.Sequential(
            nn.Linear(feature_dim, feature_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(feature_dim * 2, feature_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(feature_dim * 2, feature_dim),
            nn.LayerNorm(feature_dim)
        )
        
        # Decoder for generating output matrices from predicted features
        self.decoder = SpatialSymbolicDecoder(
            feature_dim=feature_dim,
            embed_dim=embed_dim
        )
        
        print(f"✅ DualSpatialSymbolicTransformer initialized:")
        print(f"   • Input encoder: {sum(p.numel() for p in self.input_encoder.parameters()):,} params")
        print(f"   • Output encoder: {sum(p.numel() for p in self.output_encoder.parameters()):,} params")
        print(f"   • Transformation predictor: {sum(p.numel() for p in self.transformation_predictor.parameters()):,} params")
        print(f"   • Total parameters: {sum(p.numel() for p in self.parameters()):,}")
    
    def forward(self, input_tokens, output_tokens=None, output_shapes=None, mode='train'):
        """
        Forward pass for dual encoder.
        
        Args:
            input_tokens: (batch_size, seq_len, 3) - input matrix tokens
            output_tokens: (batch_size, seq_len, 3) - output matrix tokens (for training)
            output_shapes: list of (height, width) - target output shapes (for inference)
            mode: 'train' or 'inference'
            
        Returns:
            If mode='train': predicted_output_features, actual_output_features, predicted_matrices
            If mode='inference': predicted_matrices, predicted_features
        """
        batch_size = input_tokens.size(0)
        
        # Encode input matrices
        input_features = self.input_encoder(input_tokens)
        
        # Predict output features using transformation network
        predicted_output_features = self.transformation_predictor(input_features)
        
        if mode == 'train':
            # Also encode actual output matrices for training
            actual_output_features = self.output_encoder(output_tokens)
            
            # For training, we'll decode both predicted and actual features
            # This allows us to compare feature quality
            return predicted_output_features, actual_output_features, input_features
        
        elif mode == 'inference':
            # Generate output matrices from predicted features
            predicted_matrices = []
            confidences = []
            
            for i in range(batch_size):
                target_shape = output_shapes[i]
                pred_matrix, conf = self.decoder.predict_matrix(
                    predicted_output_features[i:i+1], target_shape
                )
                predicted_matrices.append(pred_matrix.squeeze(0))
                confidences.append(conf.squeeze(0))
            
            return predicted_matrices, predicted_output_features, confidences
    
    def compute_transformation_loss(self, predicted_features, actual_features):
        """
        Compute loss between predicted and actual output features.
        Uses cosine similarity loss to encourage feature alignment.
        """
        # L2 loss between features
        mse_loss = F.mse_loss(predicted_features, actual_features)
        
        # Cosine similarity loss (encourages directional alignment)
        cos_sim = F.cosine_similarity(predicted_features, actual_features, dim=-1)
        cos_loss = (1 - cos_sim).mean()
        
        # Combined loss
        total_loss = mse_loss + 0.5 * cos_loss
        
        return total_loss, mse_loss, cos_loss

# Initialize the dual encoder system
print("🚀 Initializing Dual Encoder Architecture...")

dual_encoder = DualSpatialSymbolicTransformer(
    embed_dim=128,
    num_heads=8,
    ff_dim=512,
    num_layers=6,
    feature_dim=1024
).to(device)

# Create transformation dataset
transformation_dataset = ARCTransformationDataset(
    challenges=challenges,
    solutions=solutions,
    max_samples=300,  # Start smaller for testing
    min_size=2,
    max_size=6
)

# Create transformation dataloader
transformation_dataloader = DataLoader(
    transformation_dataset,
    batch_size=4,  # Smaller batch size due to dual encoding
    shuffle=True,
    collate_fn=collate_transformations
)

print(f"\n🎯 Dual Encoder Setup Complete!")
print(f"   • Transformation pairs: {len(transformation_dataset)}")
print(f"   • Training batches: {len(transformation_dataloader)}")
print(f"   • Total model parameters: {sum(p.numel() for p in dual_encoder.parameters()):,}")
print(f"\n✅ Ready for Step 9: Transformation learning!")

In [ ]:
# Step 9: Transformation Training - Learn Input→Output Mappings
def train_transformation_model(dual_encoder, dataloader, num_epochs=5, learning_rate=5e-5):
    """
    Train the dual encoder to learn input→output transformations.
    """
    print(f"🧠 Training Dual Encoder for ARC Transformations...")
    print(f"   • Epochs: {num_epochs}")
    print(f"   • Learning rate: {learning_rate}")
    print(f"   • Dataset size: {len(dataloader.dataset)}")
    print(f"   • Batch size: {dataloader.batch_size}")
    
    # Setup optimizer (lower learning rate for fine-tuning)
    optimizer = torch.optim.Adam(dual_encoder.parameters(), lr=learning_rate)
    
    # Training loop
    dual_encoder.train()
    epoch_losses = []
    epoch_similarities = []
    
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        epoch_mse = 0.0
        epoch_cos = 0.0
        batch_count = 0
        
        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
        
        for batch_idx, batch in enumerate(progress_bar):
            # Get batch data
            input_tokens = batch['input_tokens'].to(device)
            output_tokens = batch['output_tokens'].to(device)
            
            # Forward pass: predict output features from input
            predicted_features, actual_features, input_features = dual_encoder(
                input_tokens=input_tokens,
                output_tokens=output_tokens,
                mode='train'
            )
            
            # Compute transformation loss
            total_loss, mse_loss, cos_loss = dual_encoder.compute_transformation_loss(
                predicted_features, actual_features
            )
            
            # Backward pass
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()
            
            # Update statistics
            epoch_loss += total_loss.item()
            epoch_mse += mse_loss.item()
            epoch_cos += cos_loss.item()
            batch_count += 1
            
            # Calculate feature similarity for monitoring
            with torch.no_grad():
                similarity = F.cosine_similarity(predicted_features, actual_features, dim=-1).mean()
            
            # Update progress bar
            progress_bar.set_postfix({
                'loss': f'{total_loss.item():.4f}',
                'sim': f'{similarity.item():.3f}'
            })
        
        # Epoch statistics
        avg_loss = epoch_loss / batch_count
        avg_similarity = (epoch_mse + epoch_cos) / (2 * batch_count)
        
        epoch_losses.append(avg_loss)
        epoch_similarities.append(avg_similarity)
        
        print(f"📊 Epoch {epoch+1} Results:")
        print(f"   • Total Loss: {avg_loss:.4f}")
        print(f"   • MSE Loss: {epoch_mse/batch_count:.4f}")
        print(f"   • Cosine Loss: {epoch_cos/batch_count:.4f}")
    
    return epoch_losses, epoch_similarities

def evaluate_transformation_quality(dual_encoder, dataset, num_samples=8):
    """
    Evaluate transformation quality on sample input→output pairs.
    """
    print(f"\n🔍 Evaluating Transformation Quality on {num_samples} samples...")
    
    dual_encoder.eval()
    converter = MatrixToSequenceConverter(normalize_positions=True)
    
    total_accuracy = 0.0
    total_confidence = 0.0
    
    # Test on random samples
    sample_indices = np.random.choice(len(dataset), min(num_samples, len(dataset)), replace=False)
    
    with torch.no_grad():
        for i, idx in enumerate(sample_indices):
            sample = dataset[idx]
            input_matrix = sample['input_matrix'].numpy()
            output_matrix = sample['output_matrix'].numpy()
            
            # Convert input to tokens
            input_tokens, _ = converter.matrix_to_sequence(input_matrix)
            input_tensor = torch.tensor(input_tokens, dtype=torch.float32).unsqueeze(0).to(device)
            
            # Predict output
            predicted_matrices, predicted_features, confidences = dual_encoder(
                input_tokens=input_tensor,
                output_shapes=[output_matrix.shape],
                mode='inference'
            )
            
            # Calculate metrics
            predicted_matrix = predicted_matrices[0].cpu().numpy()
            confidence = confidences[0].cpu().numpy()
            
            accuracy = (output_matrix == predicted_matrix).mean()
            avg_confidence = confidence.mean()
            
            total_accuracy += accuracy
            total_confidence += avg_confidence
            
            # Show detailed results for first few samples
            if i < 3:
                print(f"\n📋 Sample {i+1} - Task: {sample['task_id']}")
                print(f"   Input ({input_matrix.shape}):\n{input_matrix}")
                print(f"   Expected Output ({output_matrix.shape}):\n{output_matrix}")
                print(f"   Predicted Output:\n{predicted_matrix}")
                print(f"   Accuracy: {accuracy:.3f} ({accuracy*100:.1f}%)")
                print(f"   Avg Confidence: {avg_confidence:.3f}")
    
    # Overall statistics
    avg_accuracy = total_accuracy / len(sample_indices)
    avg_confidence = total_confidence / len(sample_indices)
    
    print(f"\n🎯 Overall Transformation Quality:")
    print(f"   • Average Accuracy: {avg_accuracy:.3f} ({avg_accuracy*100:.1f}%)")
    print(f"   • Average Confidence: {avg_confidence:.3f}")
    
    return avg_accuracy, avg_confidence

def compare_architectures(dual_encoder, autoencoder, test_sample):
    """
    Compare reconstruction vs transformation capabilities.
    """
    print(f"\n🔬 Architecture Comparison Analysis...")
    
    dual_encoder.eval()
    autoencoder.eval()
    converter = MatrixToSequenceConverter(normalize_positions=True)
    
    input_matrix = test_sample['input_matrix'].numpy()
    output_matrix = test_sample['output_matrix'].numpy()
    
    with torch.no_grad():
        # Test autoencoder reconstruction on input
        input_reconstructed, _, _ = autoencoder([input_matrix], return_features=True)
        input_recon_accuracy = (input_matrix == input_reconstructed[0].cpu().numpy()).mean()
        
        # Test autoencoder reconstruction on output
        output_reconstructed, _, _ = autoencoder([output_matrix], return_features=True)
        output_recon_accuracy = (output_matrix == output_reconstructed[0].cpu().numpy()).mean()
        
        # Test dual encoder transformation
        input_tokens, _ = converter.matrix_to_sequence(input_matrix)
        input_tensor = torch.tensor(input_tokens, dtype=torch.float32).unsqueeze(0).to(device)
        
        predicted_matrices, _, _ = dual_encoder(
            input_tokens=input_tensor,
            output_shapes=[output_matrix.shape],
            mode='inference'
        )
        
        transformation_accuracy = (output_matrix == predicted_matrices[0].cpu().numpy()).mean()
    
    print(f"📊 Capability Comparison:")
    print(f"   • Autoencoder input reconstruction: {input_recon_accuracy:.3f} ({input_recon_accuracy*100:.1f}%)")
    print(f"   • Autoencoder output reconstruction: {output_recon_accuracy:.3f} ({output_recon_accuracy*100:.1f}%)")
    print(f"   • Dual encoder transformation: {transformation_accuracy:.3f} ({transformation_accuracy*100:.1f}%)")
    
    return input_recon_accuracy, output_recon_accuracy, transformation_accuracy

# Execute Transformation Training
print("🚀 Starting Transformation Learning...")

# Train the dual encoder
transformation_losses, transformation_similarities = train_transformation_model(
    dual_encoder=dual_encoder,
    dataloader=transformation_dataloader,
    num_epochs=3,  # Start with fewer epochs for testing
    learning_rate=5e-5
)

print(f"\n📈 Transformation Training Complete!")

# Evaluate transformation quality
transform_accuracy, transform_confidence = evaluate_transformation_quality(
    dual_encoder=dual_encoder,
    dataset=transformation_dataset,
    num_samples=8
)

# Compare with autoencoder capabilities
test_sample = transformation_dataset[0]
input_recon, output_recon, transform_perf = compare_architectures(
    dual_encoder, autoencoder, test_sample
)

print(f"\n🏆 Final Architecture Assessment:")
print(f"   • Spatial reasoning (reconstruction): {(input_recon + output_recon)/2*100:.1f}% avg")
print(f"   • Transformation learning: {transform_accuracy*100:.1f}%")

if transform_accuracy > 0.3:
    print(f"🎉 Excellent! The dual encoder learned meaningful transformations!")
    print(f"✅ Position-aware architecture successfully captures ARC patterns!")
elif transform_accuracy > 0.15:
    print(f"✅ Good progress! Transformations are being learned.")
    print(f"🔧 Consider more epochs or architectural refinements.")
else:
    print(f"⚠️ Transformation learning needs improvement.")
    print(f"🔧 Consider: more training data, different loss functions, or architecture adjustments.")

print(f"\n🏁 Step 9 Complete: Transformation learning finished!")
print(f"🚀 Ready to test on full ARC tasks and compare with baselines!")